In [1]:
import warnings
# Suppress all warnings
warnings.filterwarnings("ignore")

In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [10]:
from typing import Annotated, Literal, Sequence, TypedDict
from langchain_classic import hub
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field
from langgraph.graph.message import add_messages
from langgraph.prebuilt import tools_condition
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.tools import create_retriever_tool
from langgraph.graph import END, StateGraph, START
from langgraph.prebuilt import ToolNode

In [4]:
import os
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["HUGGINGFACEHUB_API_TOKEN"] = os.getenv("HUGGINGFACEHUB_API_TOKEN")
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["LANGSMITH_TRACING_V2"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_PROJECT_NAME"] = "Agentic RAG Project"

In [47]:
from langchain_tavily import TavilySearch



# Initialize the tool
tavily_tool = TavilySearch(
    max_results=5
)


In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
#embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

In [6]:
embeddings=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [42]:
from langchain_groq import ChatGroq
llm=ChatGroq(model_name="Gemma2-9b-It")

In [ ]:
#from langchain_google_genai import ChatGoogleGenerativeAI
#llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash")

In [21]:
urls=["https://www.mea.gov.in/Images/pdf1/Part1.pdf",
      "https://www.mea.gov.in/Images/pdf1/Part2.pdf"]

In [22]:
docs=[WebBaseLoader(url).load()for url in urls]

In [23]:
docs_list=[item for sublist in docs for item in sublist]

In [27]:
text_splitter=RecursiveCharacterTextSplitter.from_tiktoken_encoder(chunk_size=100,chunk_overlap=10)
texts=text_splitter.split_documents(docs_list)

In [28]:
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever

In [29]:
vectorstore=Chroma.from_documents(
    documents=texts,
    collection_name="MEA_Govt_of_india",
    embedding=embeddings)

In [31]:
# 3. Setup Lexical Keyword Retriever (BM25)
bm25_retriever = BM25Retriever.from_documents(texts)
bm25_retriever.k = 3

vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

# 4. Combine into an Ensemble Retriever
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.3, 0.7] # Balance between keyword and vector results
)

In [35]:
from flashrank import Ranker
from langchain_community.document_compressors import FlashrankRerank
from langchain_classic.retrievers import ContextualCompressionRetriever

In [36]:

# Initialize the Ranker client first
flashrank_client = Ranker(model_name="ms-marco-MiniLM-L-12-v2")

# Pass the client explicitly to bypass the validation bug
compressor = FlashrankRerank(client=flashrank_client, top_n=3)
# 2. Initialize the FlashRank compressor
#compressor = FlashrankRerank(top_n=3)

# 3. Create the Compression Retriever
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, 
    base_retriever=ensemble_retriever
)

INFO:flashrank.Ranker:Downloading ms-marco-MiniLM-L-12-v2...
ms-marco-MiniLM-L-12-v2.zip: 100%|██████████| 21.6M/21.6M [00:03<00:00, 7.29MiB/s]


In [37]:
retriever_tool = create_retriever_tool(
    name="MEA_Retriever",
    retriever=compression_retriever,
    description="Useful for answering questions about the Ministry of External Affairs of the Government of India."
)

In [48]:
tools = [retriever_tool,tavily_tool]
retrieve=ToolNode([retriever_tool])
llm_with_tools=llm.bind_tools(tools)

In [44]:
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

In [ ]:
def ai_assistant(state:AgentState):
    print("---CALL AGENT---")
    messages = state['messages']
    print(f"this is my message: {messages}")
    
    if len(messages)>1:
        response=llm.invoke(messages[-1].content)
        return {"messages": [response]}
    else:
        initial_prompt=PromptTemplate(
        template="""You are a helpful AI assistant that can answer the user's question based on the following conditions:
        1. If the user's question is about the Ministry of External Affairs of the Government of India, use the retriever tool to get relevant context to answer the question.
        2. If the user's question is about current information or data, use the Tavily Search tool to get the latest information.
        3. If the user's question is about something else, use your general knowledge to answer.""")
        initial_chain = initial_prompt | llm_with_tools
        response = initial_chain.invoke(messages)
        return {"messages": [response]}

In [50]:
class grade(BaseModel):
    binary_score:str=Field(description="Relevance score 'yes' or 'no'")

In [ ]:
def grade_documents(state:AgentState)->Literal["Output_Generator", "Query_Rewriter"]:
    llm_with_structure_op=llm.with_structured_output(grade)
    
    prompt=PromptTemplate(
        template="""You are a grader deciding if a document or context is relevant to a user’s question.
                    Here is the document/context: {context}
                    Here is the user’s question: {question}
                    If the document or context talks about or contains information related to the user’s question, mark it as relevant. 
                    Give a 'yes' or 'no' answer to show if the document is relevant to the question.""",
                    input_variables=["context", "question"]
                    )
    chain = prompt | llm_with_structure_op
    
    messages = state["messages"]
    print(f"message from the grader: {messages}")
    last_message = messages[-1]
    question = messages[0].content
    docs = last_message.content
    scored_result = chain.invoke({"question": question, "context": docs})
    score = scored_result.binary_score

    if score == "yes":
        print("---DECISION: CONTEXT RELEVANT---")
        return "generator" #this should be a node name
    else:
        print("---DECISION: CONTEXT NOT RELEVANT---")
        return "rewriter" #this should be a node name